# 🏢 Real Estate Investment Advisor: Predicting Property Profitability & Future Value
## Comprehensive Exploratory Data Analysis (EDA) & Domain Insights

This notebook explores the Indian Housing Prices dataset (`india_housing_prices.csv`, 250,000 records) across 4 major dimensions:
1. **Price & Size Analysis (Q1 - Q5)**
2. **Location-Based Analysis (Q6 - Q10)**
3. **Feature Relationship & Correlation (Q11 - Q15)**
4. **Investment, Amenities & Ownership Analysis (Q16 - Q20)**


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetics
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.titlesize': 14,
    'figure.titleweight': 'bold',
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.labelweight': 'semibold',
    'figure.dpi': 150
})

# Load engineered dataset
df = pd.read_csv('../data/processed/housing_engineered_full.csv')
print('Loaded dataset shape:', df.shape)
df.head(3)


---
## 📊 Section 1: Price & Size Analysis
Explores fundamental distributions of property pricing, sizes, price per sqft across types, relationships, and outlier behavior.


### Q1: Distribution of Property Prices
**Question:** How are property prices distributed across the overall market?


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Price_in_Lakhs'], kde=True, color='#1f77b4', bins=40)
plt.axvline(df['Price_in_Lakhs'].mean(), color='red', linestyle='--', label=f"Mean: ₹{df['Price_in_Lakhs'].mean():.2f}L")
plt.axvline(df['Price_in_Lakhs'].median(), color='green', linestyle='-', label=f"Median: ₹{df['Price_in_Lakhs'].median():.2f}L")
plt.title("Q1: Distribution of Property Prices (₹ Lakhs)")
plt.xlabel("Price (in ₹ Lakhs)")
plt.ylabel("Count")
plt.legend()
plt.show()


### Q2: Distribution of Property Sizes
**Question:** What is the spread and central tendency of property carpet/built-up sizes in square feet?


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Size_in_SqFt'], kde=True, color='#2ca02c', bins=40)
plt.axvline(df['Size_in_SqFt'].mean(), color='red', linestyle='--', label=f"Mean: {df['Size_in_SqFt'].mean():.0f} sq.ft")
plt.axvline(df['Size_in_SqFt'].median(), color='blue', linestyle='-', label=f"Median: {df['Size_in_SqFt'].median():.0f} sq.ft")
plt.title("Q2: Distribution of Property Sizes (in SqFt)")
plt.xlabel("Size (in SqFt)")
plt.ylabel("Count")
plt.legend()
plt.show()


### Q3: Price per SqFt across Property Types
**Question:** How does the rate per square foot vary between Apartments, Independent Houses, and Villas?


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(x='Property_Type', y='Price_per_SqFt_INR', data=df, palette='Set2')
plt.title("Q3: Price per SqFt (₹/SqFt) by Property Type")
plt.xlabel("Property Type")
plt.ylabel("Price per SqFt (₹)")
plt.show()


### Q4: Relationship between Property Size and Price
**Question:** How strongly does physical square footage correlate with the final asset value?


In [ ]:
sample_df = df.sample(n=3000, random_state=42)
plt.figure(figsize=(9, 5))
sns.scatterplot(x='Size_in_SqFt', y='Price_in_Lakhs', hue='Property_Type', data=sample_df, alpha=0.5, palette='tab10')
sns.regplot(x='Size_in_SqFt', y='Price_in_Lakhs', data=sample_df, scatter=False, color='black', line_kws={'linestyle':'--'})
corr = df['Size_in_SqFt'].corr(df['Price_in_Lakhs'])
plt.title(f"Q4: Size vs. Price Relationship (Pearson r = {corr:.3f})")
plt.xlabel("Size (in SqFt)")
plt.ylabel("Price (in ₹ Lakhs)")
plt.show()


### Q5: Outlier Identification in Price and Size
**Question:** Are there extreme unrepresentative outliers in price or square footage that could skew models?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df['Price_in_Lakhs'], ax=axes[0], color='#ff7f0e')
axes[0].set_title("Price (₹ Lakhs) Boxplot")
sns.boxplot(y=df['Size_in_SqFt'], ax=axes[1], color='#9467bd')
axes[1].set_title("Size (SqFt) Boxplot")
plt.tight_layout()
plt.show()


---
## 🗺️ Section 2: Location-Based Analysis
Investigates regional price per square foot across States, average property prices across Cities, locality ages, BHK distributions, and top expensive micro-markets.


### Q6: Average Price per SqFt across States
**Question:** Which Indian states command the highest average price per sq.ft?


In [ ]:
state_price_sqft = df.groupby('State')['Price_per_SqFt_INR'].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=state_price_sqft.values, y=state_price_sqft.index, palette='viridis')
plt.title("Q6: Average Price per SqFt (₹) by State")
plt.xlabel("Mean Price per SqFt (₹)")
plt.ylabel("State")
plt.show()


### Q7: Average Property Price across Cities
**Question:** What are the top 15 cities by average overall property price?


In [ ]:
city_prices = df.groupby('City')['Price_in_Lakhs'].mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(11, 5))
sns.barplot(x=city_prices.index, y=city_prices.values, palette='mako')
plt.title("Q7: Average Property Price (₹ Lakhs) across Top 15 Cities")
plt.xlabel("City")
plt.ylabel("Price (₹ Lakhs)")
plt.xticks(rotation=45, ha='right')
plt.show()


### Q8: Median Property Age across Localities
**Question:** How does building vintage/age vary across different micro-markets and localities?


In [ ]:
locality_age = df.groupby('Locality')['Age_of_Property'].median().head(20)
plt.figure(figsize=(11, 5))
sns.barplot(x=locality_age.index, y=locality_age.values, palette='cubehelix')
plt.title("Q8: Median Property Age across Sample Localities (Years)")
plt.xlabel("Locality")
plt.ylabel("Median Age (Years)")
plt.xticks(rotation=45, ha='right')
plt.show()


### Q9: BHK Configuration Distribution by Major Cities
**Question:** What is the market share of 1, 2, 3, 4, and 5 BHK units in major urban centers?


In [ ]:
top_cities = df['City'].value_counts().head(8).index
bhk_city = pd.crosstab(df[df['City'].isin(top_cities)]['City'], df['BHK'], normalize='index') * 100
bhk_city.plot(kind='bar', stacked=True, colormap='Spectral', figsize=(11, 5), edgecolor='black', linewidth=0.5)
plt.title("Q9: BHK Configuration Percentage Share across Top Cities")
plt.xlabel("City")
plt.ylabel("Share (%)")
plt.legend(title="BHK", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.show()


### Q10: Price Trends in Top 5 Most Expensive Localities
**Question:** What are the average valuations within the highest tier micro-markets?


In [ ]:
top5_localities = df.groupby('Locality')['Price_in_Lakhs'].mean().sort_values(ascending=False).head(5)
plt.figure(figsize=(8, 4.5))
sns.barplot(x=top5_localities.index, y=top5_localities.values, palette='rocket')
plt.title("Q10: Average Price (₹ Lakhs) in Top 5 Most Expensive Localities")
plt.xlabel("Locality")
plt.ylabel("Mean Price (₹ Lakhs)")
plt.show()


---
## 🔗 Section 3: Feature Relationship & Correlation
Analyzes correlations, educational infrastructure, healthcare density, furnishing status, and directional facing effects.


### Q11: Correlation Heatmap of Numeric Features
**Question:** How are structural, age, infrastructural, and pricing variables intercorrelated?


In [ ]:
num_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt_INR', 'Year_Built', 
            'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Amenity_Count']
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Q11: Numeric Features Correlation Heatmap")
plt.show()


### Q12: Nearby Schools vs. Price and Price per SqFt
**Question:** How does the count of nearby schools influence property valuation?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x='Nearby_Schools', y='Price_in_Lakhs', data=df, ax=axes[0], palette='Blues_d')
axes[0].set_title("Average Price vs Nearby Schools")
sns.barplot(x='Nearby_Schools', y='Price_per_SqFt_INR', data=df, ax=axes[1], palette='Blues_d')
axes[1].set_title("Price per SqFt vs Nearby Schools")
plt.tight_layout()
plt.show()


### Q13: Nearby Hospitals vs. Price and Price per SqFt
**Question:** What impact does proximity to healthcare infrastructure have on asset pricing?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x='Nearby_Hospitals', y='Price_in_Lakhs', data=df, ax=axes[0], palette='Greens_d')
axes[0].set_title("Average Price vs Nearby Hospitals")
sns.barplot(x='Nearby_Hospitals', y='Price_per_SqFt_INR', data=df, ax=axes[1], palette='Greens_d')
axes[1].set_title("Price per SqFt vs Nearby Hospitals")
plt.tight_layout()
plt.show()


### Q14: Property Price variation across Furnished Status
**Question:** How do prices differ across Furnished, Semi-furnished, and Unfurnished units?


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Furnished_Status', y='Price_in_Lakhs', data=df, palette='Pastel1')
plt.title("Q14: Property Price Distribution by Furnished Status")
plt.xlabel("Furnished Status")
plt.ylabel("Price (₹ Lakhs)")
plt.show()


### Q15: Price per SqFt variation across Facing Directions
**Question:** Do directional orientations (North, East, South, West / Vastu orientation) exhibit pricing differences?


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Facing', y='Price_per_SqFt_INR', data=df, palette='Set3', errorbar=None)
plt.title("Q15: Price per SqFt (₹) by Facing Direction")
plt.xlabel("Facing Direction")
plt.ylabel("Price per SqFt (₹)")
plt.show()


---
## 🏆 Section 4: Investment, Amenities & Ownership Analysis
Examines ownership breakdown, construction availability, parking premiums, amenity impact, and public transport access vs. investment attractiveness.


### Q16: Distribution of Owner Types
**Question:** What is the proportion of inventory listed by Owners, Builders, and Brokers?


In [ ]:
plt.figure(figsize=(6, 6))
owner_counts = df['Owner_Type'].value_counts()
plt.pie(owner_counts.values, labels=owner_counts.index, autopct='%1.1f%%', colors=['#4c72b0', '#55a868', '#c44e52'],
        startangle=140, explode=(0.03, 0.03, 0.03))
plt.title("Q16: Proportion of Properties by Owner Type")
plt.show()


### Q17: Distribution of Availability Status
**Question:** How is the housing inventory split between Ready to Move and Under Construction?


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='Availability_Status', data=df, palette='Accent')
plt.title("Q17: Breakdown of Property Availability Status")
plt.xlabel("Availability Status")
plt.ylabel("Count")
plt.show()


### Q18: Effect of Dedicated Parking Space on Property Price
**Question:** Does the presence of dedicated parking add a measurable valuation premium?


In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(x='Parking_Space', y='Price_in_Lakhs', data=df, palette='muted', inner='quartile')
plt.title("Q18: Effect of Dedicated Parking Space on Price")
plt.xlabel("Parking Space Available")
plt.ylabel("Price (₹ Lakhs)")
plt.show()


### Q19: Amenities Count & Diversity vs. Price per SqFt
**Question:** How does the number of modern amenities (Gym, Pool, Clubhouse, Playground, Garden) scale the square foot valuation?


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Amenity_Count', y='Price_per_SqFt_INR', data=df, palette='crest', errorbar=None)
plt.title("Q19: Impact of Amenity Count on Price per SqFt")
plt.xlabel("Number of Amenities")
plt.ylabel("Mean Price per SqFt (₹)")
plt.show()


### Q20: Public Transport Accessibility vs. Price & Investment Potential
**Question:** How strongly does high public transit accessibility correlate with classifying as a 'Good Investment'?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x='Public_Transport_Accessibility', y='Price_in_Lakhs', data=df, order=['Low', 'Medium', 'High'], 
            palette='coolwarm', ax=axes[0], errorbar=None)
axes[0].set_title("Average Price by Transport Accessibility")

sns.barplot(x='Public_Transport_Accessibility', y='Good_Investment', data=df, order=['Low', 'Medium', 'High'], 
            palette='coolwarm', ax=axes[1], errorbar=None)
axes[1].set_title("Good Investment Proportion by Transport")
plt.tight_layout()
plt.show()
